In [ ]:
import sys
import numpy
import torch
import random
import logging
import time
import json
from pathlib import Path

# Assuming the relative imports are converted into absolute ones or the modules are in the Python path
from sc_fuzzing.rl.fuzzers import Environment
from sc_fuzzing.rl.fuzzers.random import PolicyRandom, ObsRandom
from sc_fuzzing.rl.fuzzers.F_random import PolicyFRandom, ObsFRandom
from sc_fuzzing.rl.fuzzers.reinforcement import (
    PolicyReinforcementDRQN,
    ObsReinforcement,
    PolicyReinforcementPPODiscrete,
    PolicyReinforcementPPOContinuous,
    PolicyReinforcementSAC
)
from sc_fuzzing.rl.execution import Execution
# from sc_fuzzing.rl.common import set_logging
from sc_fuzzing.rl.fuzzers.reinforcement.ultis import str2bool

from sc_fuzzing.env import Env
from sc_fuzzing.env.blockchain import Ganache
from sc_fuzzing.data.dataloader import DataLoader
from sc_fuzzing.utils import set_logging
set_logging(2)

In [ ]:
from sc_fuzzing.data.fasttext import create_fasttext_w2v
create_fasttext_w2v(force=True)

In [ ]:
OUPUT_PATH = 'result/rl_sbc'
DATASET = "smartbugs_curated"
LLM = False

In [ ]:
# # Modify this dictionary to simulate CLI arguments
# sbc_metadata_df = DataLoader().get_metadata("smartbugs_curated")
# sample = sbc_metadata_df[sbc_metadata_df["name"] == "erc20"].iloc[0]
# sample

In [ ]:
def run(sample):
    arg_dict = {
        # 'execution': './execution.so',
        'proj': sample["project_path"],  # Replace with actual path
        'contract': sample['primary_contract'],  # Replace with actual contract
        'limit': 1000,
        'fuzzer': 'reinforcement_drqn',  # 'frandom', 'reinforcement_drqn', etc.
        'model': 'model_imitation',
        'seed': 1,
        'log_to_file': None,
        'v': 1,
        'train_dir': None,
        'dataset_dump_path': None,
        'address': sample['name'],
        'output_path': OUPUT_PATH,
        'mode': 'train',
        'max_episode': 50,
        'reward': None,
        'rl_model': 'model_dqn',
        'bug_rate': 0.5,
        'detect_bugs': None,
        'limit_time': 1800,
        'dvc': 'cpu',
        'EnvIdex': 0,
        'write': False,
        'render': False,
        'Loadmodel': False,
        'ModelIdex': 100,
        'Max_train_steps': int(5e6),
        'save_interval': int(100e3),
        'eval_interval': int(2.5e3),
        'update_every': 50,
        'gamma': 0.99,
        'net_width': 256,
        'a_lr': 3e-4,
        'c_lr': 3e-4,
        'batch_size': 256,
        'alpha': 0.12,
        'adaptive_alpha': True,
        'supported_llm': LLM,
        'dataset': DATASET
    }
    class Args:
        def __init__(self, arg_dict):
            self.__dict__.update(arg_dict)

    def init(args):
        random.seed(args.seed)
        # set_logging(args.v, args.log_to_file)
        torch.manual_seed(args.seed)
        numpy.random.seed(args.seed)
        sys.setrecursionlimit(8000)

    args = Args(arg_dict)
    init(args)

    start_time = time.time()
    env = Env(Ganache(), args.proj)
    env.init()

    LOG = logging.getLogger(__name__)
    LOG.info('fuzzing start')

    execution, contract_manager, account_manager = None, None, None

    if args.proj:
        execution = Execution(args.proj, env)
        contract_manager = execution.get_contracts()
        if args.contract:
            contract_manager.set_fuzz_contracts([args.contract])
        account_manager = execution.get_accounts()

    args.state_dim = 110 + 5
    policy = None
    obs = None

    if args.fuzzer == 'random':
        print('fuzzer random')
        policy = PolicyRandom(execution, contract_manager, account_manager)
        obs = ObsRandom(contract_manager, account_manager, args.dataset_dump_path)

    elif args.fuzzer == 'frandom':
        print('fuzzer frandom')
        policy = PolicyFRandom(execution, contract_manager, account_manager)
        obs = ObsFRandom(contract_manager, account_manager, args.dataset_dump_path)

    elif args.fuzzer == 'reinforcement_drqn':
        policy = PolicyReinforcementDRQN(execution, contract_manager, account_manager, args)
        if args.mode == 'train':
            print('train mode')
        policy.load_model()
        obs = ObsReinforcement(contract_manager, account_manager, args.dataset_dump_path)

    elif args.fuzzer == 'reinforcement_ppo_discrete':
        policy = PolicyReinforcementPPODiscrete(execution, contract_manager, account_manager, args)
        if args.mode == 'train':
            print('train mode')
        policy.load_model()
        obs = ObsReinforcement(contract_manager, account_manager, args.dataset_dump_path)

    elif args.fuzzer == 'reinforcement_ppo_continuous':
        policy = PolicyReinforcementPPOContinuous(execution, contract_manager, account_manager, args)
        if args.mode == 'train':
            print('train mode')
        policy.load_model()
        obs = ObsReinforcement(contract_manager, account_manager, args.dataset_dump_path)

    elif args.fuzzer == 'reinforcement_sac':
        policy = PolicyReinforcementSAC(execution, contract_manager, account_manager, args)
        if args.mode == 'train':
            print('train mode')
        policy.load_model()
        obs = ObsReinforcement(contract_manager, account_manager, args.dataset_dump_path)
    environment = Environment(args.limit, args.seed, args.max_episode, start_time)

    if args.fuzzer == 'reinforcement_drqn':
        result = environment.MADFuzz_drqn(policy, obs, start_time, args)
    elif args.fuzzer == 'reinforcement_ppo_discrete':
        result = environment.MADFuzz_ppo_discrete(policy, obs, start_time, args)
    elif args.fuzzer == 'reinforcement_ppo_continuous':
        result = environment.MADFuzz_ppo_continuous(policy, obs, start_time, args)
    elif args.fuzzer == 'reinforcement_sac':
        result = environment.MADFuzz_sac(policy, obs, start_time, args)
    else:
        result = environment.fuzz_loop(policy, obs)

    end_time = time.time()
    result['fuzzer'] = args.fuzzer
    result['start_time'] = start_time
    result['end_time'] = end_time
    result['time'] = end_time - start_time
    result['final'] = obs.stat.export_result()

    if args.output_path:
        with open(f"{args.output_path}/{args.address}.json", "w") as f:
            json.dump(result, f)

    print("Fuzzing complete")
    env.stop_ganache()

In [ ]:
# from concurrent.futures import ThreadPoolExecutor
# def run_if_needed(sample):
#     name = sample['name']
#     result_path = Path(f"result/rl_llm_sbc/{name}.json")

#     if result_path.exists():
#         print(f"{name} is already run. Skip...")
#         return
    
#     print(f"Start Fuzzing for {name}")
#     run(sample)  # Assuming this performs side-effects and doesn't return

# # Load your metadata
# sbc_metadata_df = DataLoader().get_metadata("smartbugs_curated")

# # 🧵 Multithreading
# with ThreadPoolExecutor(max_workers=4) as executor:
#     for _, sample in sbc_metadata_df.iterrows():
#         executor.submit(run_if_needed, sample)

In [ ]:
df = DataLoader().get_metadata(DATASET)
Path(OUPUT_PATH).mkdir(exist_ok=True)
for _, sample in df.iterrows():
    if Path(OUPUT_PATH, f"{sample['name']}.json").exists():
        print(f"{sample['name']} is already run. Skip...")
        continue
    print(f"Start Fuzzing for {sample['name']}")
    run(sample)

In [ ]:
OUPUT_PATH = 'result/rl_sbw'
DATASET = "smartbugs_wild"
LLM = True
df = DataLoader().get_metadata(DATASET)
Path(OUPUT_PATH).mkdir(exist_ok=True)
for _, sample in df.iterrows():
    if Path(OUPUT_PATH, f"{sample['name']}.json").exists():
        print(f"{sample['name']} is already run. Skip...")
        continue
    print(f"Start Fuzzing for {sample['name']}")
    run(sample)

In [ ]:

# arg_dict = {
#     # 'execution': './execution.so',
#     'proj': sample["project_path"],  # Replace with actual path
#     'contract': sample['primary_contract'],  # Replace with actual contract
#     'limit': 100,
#     'fuzzer': 'reinforcement_drqn',  # 'frandom', 'reinforcement_drqn', etc.
#     'model': 'model_imitation',
#     'seed': 1,
#     'log_to_file': None,
#     'v': 1,
#     'train_dir': None,
#     'dataset_dump_path': None,
#     'address': '0xYourContractAddress',
#     'output_path': './result',
#     'mode': 'train',
#     'max_episode': 50,
#     'reward': None,
#     'rl_model': 'model_dqn',
#     'bug_rate': 0.5,
#     'detect_bugs': None,
#     'limit_time': 1800,
#     'dvc': 'cpu',
#     'EnvIdex': 0,
#     'write': False,
#     'render': False,
#     'Loadmodel': False,
#     'ModelIdex': 100,
#     'Max_train_steps': int(5e6),
#     'save_interval': int(100e3),
#     'eval_interval': int(2.5e3),
#     'update_every': 50,
#     'gamma': 0.99,
#     'net_width': 256,
#     'a_lr': 3e-4,
#     'c_lr': 3e-4,
#     'batch_size': 256,
#     'alpha': 0.12,
#     'adaptive_alpha': True
# }

In [ ]:
# class Args:
#     def __init__(self, arg_dict):
#         self.__dict__.update(arg_dict)

# def init(args):
#     random.seed(args.seed)
#     set_logging(args.v, args.log_to_file)
#     torch.manual_seed(args.seed)
#     numpy.random.seed(args.seed)
#     sys.setrecursionlimit(8000)

# args = Args(arg_dict)
# init(args)

In [ ]:
# start_time = time.time()
# env = Env(Ganache(), args.proj)
# env.init()

# LOG = logging.getLogger(__name__)
# LOG.info('fuzzing start')

# execution, contract_manager, account_manager = None, None, None

# if args.proj:
#     execution = Execution(args.proj, env)
#     contract_manager = execution.get_contracts()
#     if args.contract:
#         contract_manager.set_fuzz_contracts([args.contract])
#     account_manager = execution.get_accounts()

# args.state_dim = 110 + 5
# policy = None
# obs = None

# if args.fuzzer == 'random':
#     print('fuzzer random')
#     policy = PolicyRandom(execution, contract_manager, account_manager)
#     obs = ObsRandom(contract_manager, account_manager, args.dataset_dump_path)

# elif args.fuzzer == 'frandom':
#     print('fuzzer frandom')
#     policy = PolicyFRandom(execution, contract_manager, account_manager)
#     obs = ObsFRandom(contract_manager, account_manager, args.dataset_dump_path)

# elif args.fuzzer == 'reinforcement_drqn':
#     policy = PolicyReinforcementDRQN(execution, contract_manager, account_manager, args)
#     if args.mode == 'train':
#         print('train mode')
#     policy.load_model()
#     obs = ObsReinforcement(contract_manager, account_manager, args.dataset_dump_path)

# elif args.fuzzer == 'reinforcement_ppo_discrete':
#     policy = PolicyReinforcementPPODiscrete(execution, contract_manager, account_manager, args)
#     if args.mode == 'train':
#         print('train mode')
#     policy.load_model()
#     obs = ObsReinforcement(contract_manager, account_manager, args.dataset_dump_path)

# elif args.fuzzer == 'reinforcement_ppo_continuous':
#     policy = PolicyReinforcementPPOContinuous(execution, contract_manager, account_manager, args)
#     if args.mode == 'train':
#         print('train mode')
#     policy.load_model()
#     obs = ObsReinforcement(contract_manager, account_manager, args.dataset_dump_path)

# elif args.fuzzer == 'reinforcement_sac':
#     policy = PolicyReinforcementSAC(execution, contract_manager, account_manager, args)
#     if args.mode == 'train':
#         print('train mode')
#     policy.load_model()
#     obs = ObsReinforcement(contract_manager, account_manager, args.dataset_dump_path)

In [ ]:
# policy = None
# obs = None

# if args.fuzzer == 'random':
#     print('fuzzer random')
#     policy = PolicyRandom(execution, contract_manager, account_manager)
#     obs = ObsRandom(contract_manager, account_manager, args.dataset_dump_path)

# elif args.fuzzer == 'frandom':
#     print('fuzzer frandom')
#     policy = PolicyFRandom(execution, contract_manager, account_manager)
#     obs = ObsFRandom(contract_manager, account_manager, args.dataset_dump_path)

# elif args.fuzzer == 'reinforcement_drqn':
#     policy = PolicyReinforcementDRQN(execution, contract_manager, account_manager, args)
#     if args.mode == 'train':
#         print('train mode')
#     policy.load_model()
#     obs = ObsReinforcement(contract_manager, account_manager, args.dataset_dump_path)

# elif args.fuzzer == 'reinforcement_ppo_discrete':
#     policy = PolicyReinforcementPPODiscrete(execution, contract_manager, account_manager, args)
#     if args.mode == 'train':
#         print('train mode')
#     policy.load_model()
#     obs = ObsReinforcement(contract_manager, account_manager, args.dataset_dump_path)

# elif args.fuzzer == 'reinforcement_ppo_continuous':
#     policy = PolicyReinforcementPPOContinuous(execution, contract_manager, account_manager, args)
#     if args.mode == 'train':
#         print('train mode')
#     policy.load_model()
#     obs = ObsReinforcement(contract_manager, account_manager, args.dataset_dump_path)

# elif args.fuzzer == 'reinforcement_sac':
#     policy = PolicyReinforcementSAC(execution, contract_manager, account_manager, args)
#     if args.mode == 'train':
#         print('train mode')
#     policy.load_model()
#     obs = ObsReinforcement(contract_manager, account_manager, args.dataset_dump_path)

In [ ]:
# environment = Environment(args.limit, args.seed, args.max_episode, start_time)

# if args.fuzzer == 'reinforcement_drqn':
#     result = environment.MADFuzz_drqn(policy, obs, start_time, args)
# elif args.fuzzer == 'reinforcement_ppo_discrete':
#     result = environment.MADFuzz_ppo_discrete(policy, obs, start_time, args)
# elif args.fuzzer == 'reinforcement_ppo_continuous':
#     result = environment.MADFuzz_ppo_continuous(policy, obs, start_time, args)
# elif args.fuzzer == 'reinforcement_sac':
#     result = environment.MADFuzz_sac(policy, obs, start_time, args)
# else:
#     result = environment.fuzz_loop(policy, obs)

# end_time = time.time()

In [ ]:
# result['fuzzer'] = args.fuzzer
# result['start_time'] = start_time
# result['end_time'] = end_time
# result['time'] = end_time - start_time
# result['final'] = obs.stat.export_result()

# if args.output_path:
#     with open(f"{args.output_path}/{args.address}.json", "w") as f:
#         json.dump(result, f)

# print("Fuzzing complete")
# env.stop()